# 個々の予測の根拠を可視化する (LIME)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/09_classification/lime_explain.ipynb)

NLP クックブック 09-13 ／ テキスト分類

ある文書がなぜそのクラスに分類されたのか、どの単語が効いたのかを、モデルの中身に関係なく説明したい。

In [ ]:
!pip install -q lime

In [ ]:
from lime.lime_text import LimeTextExplainer
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

cats = ["alt.atheism", "soc.religion.christian"]
train = fetch_20newsgroups(subset="train", categories=cats)   # あえてヘッダを残す
test = fetch_20newsgroups(subset="test", categories=cats)
clf = make_pipeline(TfidfVectorizer(sublinear_tf=True), LogisticRegression(C=10, max_iter=1000)).fit(train.data, train.target)
print("正解率:", round(clf.score(test.data, test.target), 3))

In [ ]:
explainer = LimeTextExplainer(class_names=["atheism", "christian"], random_state=0)
i = 83
exp = explainer.explain_instance(test.data[i], clf.predict_proba, num_features=10, num_samples=2000)
print("正解:", cats[test.target[i]], " 予測確率:", clf.predict_proba([test.data[i]]).round(2))
for word, weight in exp.as_list():
    print(f"{weight:+.3f}  {word}")

In [ ]:
exp.show_in_notebook(text=False)   # Colab では棒グラフと色付きテキストで表示される

- LIME は、単語をランダムに消した文を多数作って予測の変化を調べ、局所的な線形モデルで近似する。分類器が何であっても (BERT でも) 使える。
- この例では `Posting`、`Host`、`edu` などメールのヘッダ由来の語が効いている。高い正解率が本質的でない手がかりによるものだと分かる、有名な例。
- 日本語は `LimeTextExplainer(split_expression=lambda s: 分かち書き(s).split(), bow=False)` のように分割方法を渡す。